<a href="https://colab.research.google.com/github/Aman21adu/aastu-week3-spam-filter/blob/main/spam_filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SMS Spam Detection — Robustness Evaluation

**Classical machine learning | TF-IDF + Multinomial Naive Bayes**

**Objective.** Train a binary spam classifier on the SMS Spam Collection, evaluate it on held-out messages, and challenge it with five original messages that probe spelling tricks, negation, formal language and misleading context.

## 1. Dataset

**Source:** [SMS Spam Collection — UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) · [DOI: 10.24432/C5CC84](https://doi.org/10.24432/C5CC84)  
**Creators:** Tiago A. Almeida and José María Gómez Hidalgo · **License:** CC BY 4.0  
**Labels:** `ham` (legitimate) and `spam` (unsolicited). The original dataset is retrieved at runtime and is not included in this repository.

In [1]:
# Only install a dependency when it is missing from the environment.
import importlib.util
import subprocess
import sys

needed = {
    'numpy': 'numpy>=1.26,<3',
    'pandas': 'pandas>=2,<3',
    'sklearn': 'scikit-learn>=1.4,<2',
}
missing = [spec for module, spec in needed.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Required Python packages are available.')

Required Python packages are available.


In [2]:
import csv
import io
import urllib.request
import zipfile
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

SOURCES = [
    ('UCI Machine Learning Repository',
     'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip', 'zip'),
    ('Public mirror of the SMS Spam Collection',
     'https://raw.githubusercontent.com/sbiguzzi/data607project4/main/SMSSpamCollection.txt', 'text'),
]

def get_sms_data():
    errors = []
    for source, url, kind in SOURCES:
        try:
            request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(request, timeout=50) as response:
                content = response.read()
            if kind == 'zip':
                with zipfile.ZipFile(io.BytesIO(content)) as archive:
                    content = archive.read('SMSSpamCollection')
            for encoding in ('utf-8', 'latin-1'):
                try:
                    frame = pd.read_csv(
                        io.BytesIO(content), sep='\t', header=None,
                        names=['label', 'message'], encoding=encoding,
                        quoting=csv.QUOTE_NONE, on_bad_lines='error'
                    )
                    break
                except UnicodeDecodeError:
                    continue
            if not {'label', 'message'}.issubset(frame.columns) or len(frame) < 5000:
                raise ValueError('Unexpected dataset schema or incomplete download.')
            return frame, source
        except Exception as exc:
            errors.append(f'{source}: {type(exc).__name__}: {exc}')
    raise RuntimeError('Could not retrieve the SMS Spam Collection.\n' + '\n'.join(errors))

raw, dataset_source = get_sms_data()
data = raw.dropna(subset=['label', 'message']).copy()
data['label'] = data['label'].astype(str).str.strip().str.lower()
data['message'] = data['message'].astype(str).str.strip()
data = data[data['label'].isin(['ham', 'spam']) & data['message'].ne('')]
data = data.drop_duplicates(subset=['message']).reset_index(drop=True)
assert set(data['label']) == {'ham', 'spam'}
assert data['message'].is_unique
print(f'Dataset: {dataset_source}')
print(f'Original records: {len(raw):,} | Unique usable records: {len(data):,}')
display(data['label'].value_counts().rename_axis('Label').reset_index(name='Messages'))

Dataset: UCI Machine Learning Repository
Original records: 5,574 | Unique usable records: 5,160


,Label,Messages
0,ham,4518
1,spam,642


## 2. Model and held-out evaluation

An 80/20 **stratified** split is made before vectorization. TF-IDF vocabulary and Naive Bayes parameters are fitted **only on the training partition**. The held-out test set is used for final performance evaluation; the five challenge messages are separate, researcher-written probes.

In [3]:
SEED = 42
X_train, X_test, y_train, y_test = train_test_split(
    data['message'], data['label'], test_size=0.20,
    stratify=data['label'], random_state=SEED
)

vec = TfidfVectorizer()
model = MultinomialNB()
X_train_tfidf = vec.fit_transform(X_train)
model.fit(X_train_tfidf, y_train)

train_pred = model.predict(X_train_tfidf)
test_pred = model.predict(vec.transform(X_test))

scores = {
    'Training accuracy': accuracy_score(y_train, train_pred),
    'Held-out test accuracy': accuracy_score(y_test, test_pred),
    'Spam precision': precision_score(y_test, test_pred, pos_label='spam', zero_division=0),
    'Spam recall': recall_score(y_test, test_pred, pos_label='spam', zero_division=0),
    'Spam F1': f1_score(y_test, test_pred, pos_label='spam', zero_division=0),
}
score_table = pd.DataFrame({'Metric': list(scores), 'Value': [f'{v:.4f}' for v in scores.values()]})
print(f'Train: {len(X_train):,} messages | Test: {len(X_test):,} messages')
display(score_table)

cm = confusion_matrix(y_test, test_pred, labels=['ham', 'spam'])
cm_table = pd.DataFrame(cm, index=['Actual ham', 'Actual spam'],
                        columns=['Predicted ham', 'Predicted spam'])
print('Held-out confusion matrix:')
display(cm_table)

print('False positives (legitimate messages marked spam):', int(cm[0, 1]))
print('False negatives (spam messages missed):', int(cm[1, 0]))
assert 0 <= scores['Held-out test accuracy'] <= 1

Train: 4,128 messages | Test: 1,032 messages


,Metric,Value
0,Training accuracy,0.9692
1,Held-out test accuracy,0.9506
2,Spam precision,1.0000
3,Spam recall,0.6016
4,Spam F1,0.7512


Held-out confusion matrix:


,Predicted ham,Predicted spam
Actual ham,904,0
Actual spam,51,77


False positives (legitimate messages marked spam): 0
False negatives (spam messages missed): 51


## 3. Five original challenge messages

The intended class is assigned from the message's scenario, independently of the classifier. These probes are **not** used for training or for computing test accuracy. A challenge is counted as successful only when the model's predicted class differs from its intended class.

In [4]:
attempts = pd.DataFrame([
    {
        'ID': 'A1', 'Technique': 'Obfuscated spelling', 'Should be': 'spam',
        'My message': 'Cl@im your fr33 g1ft c@rd n0w! Visit the prize desk to collect it.',
        'Risk': 'Replacing letters with digits or symbols may hide learned spam tokens.'
    },
    {
        'ID': 'A2', 'Technique': 'Negation', 'Should be': 'ham',
        'My message': 'Not free, just confirming lunch at 1pm tomorrow. See you there!',
        'Risk': 'The word free appears, but the message explicitly negates an offer.'
    },
    {
        'ID': 'A3', 'Technique': 'Formal phishing tone', 'Should be': 'spam',
        'My message': 'Dear customer, a routine account verification is pending. Please send your password to the support representative to prevent an interruption.',
        'Risk': 'Polite service language may resemble ordinary transactional messages.'
    },
    {
        'ID': 'A4', 'Technique': 'Benign prize vocabulary', 'Should be': 'ham',
        'My message': 'Our class won a free prize in the campus raffle! I am bringing the cake tomorrow.',
        'Risk': 'Prize and free appear in a legitimate personal announcement.'
    },
    {
        'ID': 'A5', 'Technique': 'Low-pressure billing pretext', 'Should be': 'spam',
        'My message': 'Hello, we noticed a payment discrepancy on your invoice. Kindly reply with your full card details to finish the billing review.',
        'Risk': 'A billing pretext avoids the usual winning and reward vocabulary.'
    },
])

attempts['Model said'] = model.predict(vec.transform(attempts['My message']))
attempts['Outcome'] = np.where(attempts['Model said'] != attempts['Should be'],
                               'Fooled model', 'Correct')

def interpret_attempt(row):
    prefix = 'Possible reason for this error: ' if row['Outcome'] == 'Fooled model' else 'No error here; tested risk: '
    return prefix + row['Risk']

attempts['Why / analysis'] = attempts.apply(interpret_attempt, axis=1)

result_cols = ['ID', 'My message', 'Model said', 'Should be', 'Technique', 'Outcome', 'Why / analysis']
display(attempts[result_cols].style.hide(axis='index'))
fooled_count = int((attempts['Outcome'] == 'Fooled model').sum())
print(f'Challenge messages that fooled the model: {fooled_count} of {len(attempts)}')
assert len(attempts) == 5
assert (attempts['Model said'].isin(['ham', 'spam'])).all()

ID,My message,Model said,Should be,Technique,Outcome,Why / analysis
A1,Cl@im your fr33 g1ft c@rd n0w! Visit the prize desk to collect it.,ham,spam,Obfuscated spelling,Fooled model,Possible reason for this error: Replacing letters with digits or symbols may hide learned spam tokens.
A2,"Not free, just confirming lunch at 1pm tomorrow. See you there!",ham,ham,Negation,Correct,"No error here; tested risk: The word free appears, but the message explicitly negates an offer."
A3,"Dear customer, a routine account verification is pending. Please send your password to the support representative to prevent an interruption.",ham,spam,Formal phishing tone,Fooled model,Possible reason for this error: Polite service language may resemble ordinary transactional messages.
A4,Our class won a free prize in the campus raffle! I am bringing the cake tomorrow.,ham,ham,Benign prize vocabulary,Correct,No error here; tested risk: Prize and free appear in a legitimate personal announcement.
A5,"Hello, we noticed a payment discrepancy on your invoice. Kindly reply with your full card details to finish the billing review.",ham,spam,Low-pressure billing pretext,Fooled model,Possible reason for this error: A billing pretext avoids the usual winning and reward vocabulary.


Challenge messages that fooled the model: 3 of 5


## 4. Findings and limitations

A correct classification does not prove robustness to that technique, and a failure explanation is a plausible interpretation rather than a proven causal attribution. These five crafted messages are a qualitative stress test, **not** an independent estimate of performance in real-world deployment.

In [5]:
fp = int(cm[0, 1])
fn = int(cm[1, 0])

lesson_one = (
    f'The TF-IDF / Multinomial Naive Bayes baseline achieved '
    f'{scores["Held-out test accuracy"]:.2%} accuracy on held-out SMS messages, '
    f'with {fp} false positives and {fn} false negatives.'
)
lesson_two = (
    f'On five independently written challenge messages, {fooled_count} were misclassified; '
    'word-based features can be sensitive to obfuscation, negation and the context of ordinary business language.'
)

display(Markdown('**What I learned**\n\n' + lesson_one + '\n\n' + lesson_two))

**What I learned**

The TF-IDF / Multinomial Naive Bayes baseline achieved 95.06% accuracy on held-out SMS messages, with 0 false positives and 51 false negatives.

On five independently written challenge messages, 3 were misclassified; word-based features can be sensitive to obfuscation, negation and the context of ordinary business language.

## 5. Results report

The report below is generated directly from the executed model and challenge predictions.

In [6]:
from pathlib import Path

def safe_cell(text):
    return ' '.join(str(text).replace('|', r'\|').split())

challenge_rows = []
for _, r in attempts.iterrows():
    challenge_rows.append(
        '| ' + ' | '.join(safe_cell(r[key]) for key in
            ['My message', 'Model said', 'Should be', 'Technique', 'Outcome', 'Why / analysis']) + ' |'
    )

metric_rows = '\n'.join(
    f'| {safe_cell(row["Metric"])} | {safe_cell(row["Value"])} |'
    for _, row in score_table.iterrows()
)

report = f"""# SMS Spam Detector — Robustness Evaluation

## Objective
Train a classical text classifier to distinguish spam from legitimate SMS messages, measure its performance on held-out data, and examine five challenging messages intended to reveal its limitations.

## Dataset
**SMS Spam Collection**, UCI Machine Learning Repository (Almeida and Gómez Hidalgo, 2011), CC BY 4.0. The original dataset is downloaded when the notebook runs; it is **not included in this repository**.

- Dataset: https://archive.ics.uci.edu/dataset/228/sms+spam+collection
- DOI: https://doi.org/10.24432/C5CC84
- Raw records loaded: {len(raw):,}
- Unique, usable messages: {len(data):,}

## Method
The data was checked for empty or duplicate messages, and split into **80% training / 20% held-out testing**, stratified by class with random seed 42. `TfidfVectorizer` was fitted only to the training text, then a `MultinomialNB` classifier was trained on the TF-IDF features. The challenge messages were authored separately and were not used to fit the model or compute held-out accuracy.

## Held-out performance
Train set: {len(X_train):,} messages. Test set: {len(X_test):,} messages.

| Metric | Value |
|---|---:|
{metric_rows}

**Test confusion matrix** (rows = actual; columns = predicted):

|  | Predicted ham | Predicted spam |
|---|---:|---:|
| Actual ham | {cm[0,0]} | {cm[0,1]} |
| Actual spam | {cm[1,0]} | {cm[1,1]} |

## Five challenge messages

| My message | Model said | Should be | Trick | Outcome | Why / analysis |
|---|---|---|---|---|---|
{chr(10).join(challenge_rows)}

**Challenge errors:** {fooled_count} of 5 messages were misclassified. "Why / analysis" proposes possible explanations, not proven causes.

## What I learned
{lesson_one}

{lesson_two}

## Limitations
The model uses lexical TF-IDF features rather than a deep language representation. The dataset is older SMS text, so test performance may not reflect modern messaging. Five hand-written probes demonstrate specific sensitivities but cannot estimate production reliability; false positives on legitimate messages are especially important to monitor.
"""

Path('README.md').write_text(report, encoding='utf-8')
display(Markdown(report))
print('Generated README.md from the actual model results.')

# SMS Spam Detector — Robustness Evaluation

## Objective
Train a classical text classifier to distinguish spam from legitimate SMS messages, measure its performance on held-out data, and examine five challenging messages intended to reveal its limitations.

## Dataset
**SMS Spam Collection**, UCI Machine Learning Repository (Almeida and Gómez Hidalgo, 2011), CC BY 4.0. The original dataset is downloaded when the notebook runs; it is **not included in this repository**.

- Dataset: https://archive.ics.uci.edu/dataset/228/sms+spam+collection
- DOI: https://doi.org/10.24432/C5CC84
- Raw records loaded: 5,574
- Unique, usable messages: 5,160

## Method
The data was checked for empty or duplicate messages, and split into **80% training / 20% held-out testing**, stratified by class with random seed 42. `TfidfVectorizer` was fitted only to the training text, then a `MultinomialNB` classifier was trained on the TF-IDF features. The challenge messages were authored separately and were not used to fit the model or compute held-out accuracy.

## Held-out performance
Train set: 4,128 messages. Test set: 1,032 messages.

| Metric | Value |
|---|---:|
| Training accuracy | 0.9692 |
| Held-out test accuracy | 0.9506 |
| Spam precision | 1.0000 |
| Spam recall | 0.6016 |
| Spam F1 | 0.7512 |

**Test confusion matrix** (rows = actual; columns = predicted):

|  | Predicted ham | Predicted spam |
|---|---:|---:|
| Actual ham | 904 | 0 |
| Actual spam | 51 | 77 |

## Five challenge messages

| My message | Model said | Should be | Trick | Outcome | Why / analysis |
|---|---|---|---|---|---|
| Cl@im your fr33 g1ft c@rd n0w! Visit the prize desk to collect it. | ham | spam | Obfuscated spelling | Fooled model | Possible reason for this error: Replacing letters with digits or symbols may hide learned spam tokens. |
| Not free, just confirming lunch at 1pm tomorrow. See you there! | ham | ham | Negation | Correct | No error here; tested risk: The word free appears, but the message explicitly negates an offer. |
| Dear customer, a routine account verification is pending. Please send your password to the support representative to prevent an interruption. | ham | spam | Formal phishing tone | Fooled model | Possible reason for this error: Polite service language may resemble ordinary transactional messages. |
| Our class won a free prize in the campus raffle! I am bringing the cake tomorrow. | ham | ham | Benign prize vocabulary | Correct | No error here; tested risk: Prize and free appear in a legitimate personal announcement. |
| Hello, we noticed a payment discrepancy on your invoice. Kindly reply with your full card details to finish the billing review. | ham | spam | Low-pressure billing pretext | Fooled model | Possible reason for this error: A billing pretext avoids the usual winning and reward vocabulary. |

**Challenge errors:** 3 of 5 messages were misclassified. "Why / analysis" proposes possible explanations, not proven causes.

## What I learned
The TF-IDF / Multinomial Naive Bayes baseline achieved 95.06% accuracy on held-out SMS messages, with 0 false positives and 51 false negatives.

On five independently written challenge messages, 3 were misclassified; word-based features can be sensitive to obfuscation, negation and the context of ordinary business language.

## Limitations
The model uses lexical TF-IDF features rather than a deep language representation. The dataset is older SMS text, so test performance may not reflect modern messaging. Five hand-written probes demonstrate specific sensitivities but cannot estimate production reliability; false positives on legitimate messages are especially important to monitor.


Generated README.md from the actual model results.
